## Importing necessary libraries

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from tqdm import tqdm

## Import Dataset

In [3]:
df=pd.read_csv('./sample_data/fashion.csv')

# EDA

In [4]:
df.head()

,ProductId,Gender,Category,SubCategory,ProductType,Colour,Usage,ProductTitle,Image,ImageURL
0,42419,Girls,Apparel,Topwear,Tops,White,Casual,Gini and Jony Girls Knit White Top,42419.jpg,http://assets.myntassets.com/v1/images/style/p...
1,34009,Girls,Apparel,Topwear,Tops,Black,Casual,Gini and Jony Girls Black Top,34009.jpg,http://assets.myntassets.com/v1/images/style/p...
2,40143,Girls,Apparel,Topwear,Tops,Blue,Casual,Gini and Jony Girls Pretty Blossom Blue Top,40143.jpg,http://assets.myntassets.com/v1/images/style/p...
3,23623,Girls,Apparel,Topwear,Tops,Pink,Casual,Doodle Kids Girls Pink I love Shopping Top,23623.jpg,http://assets.myntassets.com/v1/images/style/p...
4,47154,Girls,Apparel,Bottomwear,Capris,Black,Casual,Gini and Jony Girls Black Capris,47154.jpg,http://assets.myntassets.com/v1/images/style/p...


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2906 entries, 0 to 2905
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   ProductId     2906 non-null   int64 
 1   Gender        2906 non-null   object
 2   Category      2906 non-null   object
 3   SubCategory   2906 non-null   object
 4   ProductType   2906 non-null   object
 5   Colour        2906 non-null   object
 6   Usage         2906 non-null   object
 7   ProductTitle  2906 non-null   object
 8   Image         2906 non-null   object
 9   ImageURL      2906 non-null   object
dtypes: int64(1), object(9)
memory usage: 227.2+ KB


The `TripletDataset` expects a specific directory structure with `anchor`, `positive`, and `negative` subdirectories. The previous error indicated that these directories were not found. I will create the necessary empty directory structure here.

In [16]:
import os

def create_triplet_dirs(base_path):
    """Creates the required anchor, positive, and negative subdirectories."""
    for sub_dir_name in ['anchor', 'positive', 'negative']:
        path = os.path.join(base_path, sub_dir_name)
        os.makedirs(path, exist_ok=True)
        print(f"Created directory: {path}")

# Define the base paths for training and testing data
train_base_path = './train/Footwear/Men/'
test_base_path = './test/Footwear/Men/'

print("Creating directories for training data...")
create_triplet_dirs(train_base_path)

print("\nCreating directories for testing data...")
create_triplet_dirs(test_base_path)

print("\nDirectory structure created.")

Creating directories for training data...
Created directory: ./train/Footwear/Men/anchor
Created directory: ./train/Footwear/Men/positive
Created directory: ./train/Footwear/Men/negative

Creating directories for testing data...
Created directory: ./test/Footwear/Men/anchor
Created directory: ./test/Footwear/Men/positive
Created directory: ./test/Footwear/Men/negative

Directory structure created.


In [17]:
class L2Normalize(nn.Module):
    def __init__(self, p=2, dim=1, eps=1e-12):
        super().__init__()
        self.p = p
        self.dim = dim
        self.eps = eps

    def forward(self, x):
        return nn.functional.normalize(x, p=self.p, dim=self.dim, eps=self.eps)

In [18]:
def get_resnet50_encoder(embedding_dim=512, pretrained=True, train_backbone=False):
    resnet = models.resnet50(pretrained=pretrained)
    modules = list(resnet.children())[:-1]  # Remove last FC layer
    backbone = nn.Sequential(*modules)

    model = nn.Sequential(
        backbone,
        nn.Flatten(),
        nn.Linear(2048, embedding_dim),
        L2Normalize()  # L2 normalize embeddings
    )

    # Optionally freeze backbone
    if not train_backbone:
        for param in backbone.parameters():
            param.requires_grad = False

    return model

In [19]:
# ===== 2. Dataset & DataLoader =====
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [20]:
import os
import shutil

def rename_triplet_files(base_dir):
    """
    Renames files in 'positive' and 'negative' subdirectories to match
    corresponding files in the 'anchor' subdirectory, assuming a positional match.

    Args:
        base_dir (str): The root directory containing 'anchor', 'positive',
                        and 'negative' subdirectories (e.g., './train/Footwear/Men/').
    """
    anchor_dir = os.path.join(base_dir, 'anchor')
    positive_dir = os.path.join(base_dir, 'positive')
    negative_dir = os.path.join(base_dir, 'negative')

    if not all(os.path.isdir(d) for d in [anchor_dir, positive_dir, negative_dir]):
        print(f"Error: One or more triplet directories not found in {base_dir}")
        return

    # Get sorted lists of files from each directory
    anchor_files = sorted([f for f in os.listdir(anchor_dir) if os.path.isfile(os.path.join(anchor_dir, f))])
    positive_files = sorted([f for f in os.listdir(positive_dir) if os.path.isfile(os.path.join(positive_dir, f))])
    negative_files = sorted([f for f in os.listdir(negative_dir) if os.path.isfile(os.path.join(negative_dir, f))])

    if not (len(anchor_files) == len(positive_files) == len(negative_files)):
        print("\nWARNING: Number of files in anchor, positive, and negative directories do not match.")
        print(f"Anchor files: {len(anchor_files)}, Positive files: {len(positive_files)}, Negative files: {len(negative_files)}")
        print("Proceeding with the minimum count, but this might lead to incorrect pairings or leave some files unrenamed.")
        min_count = min(len(anchor_files), len(positive_files), len(negative_files))
    else:
        min_count = len(anchor_files)

    print(f"\nAttempting to rename {min_count} triplets in '{base_dir}'...")
    renamed_count = 0
    for i in range(min_count):
        anchor_name = anchor_files[i]
        old_positive_name = positive_files[i]
        old_negative_name = negative_files[i]

        # Define new paths for positive and negative files
        new_positive_path = os.path.join(positive_dir, anchor_name)
        new_negative_path = os.path.join(negative_dir, anchor_name)

        # Define current paths for positive and negative files
        old_positive_path = os.path.join(positive_dir, old_positive_name)
        old_negative_path = os.path.join(negative_dir, old_negative_name)

        # Rename positive file
        if old_positive_name != anchor_name: # Only rename if name is different
            if os.path.exists(new_positive_path):
                print(f"  Skipping positive rename for '{anchor_name}': Target path '{new_positive_path}' already exists. Move/delete it first if you want to replace.")
            else:
                try:
                    os.rename(old_positive_path, new_positive_path)
                    print(f"  Renamed positive: '{old_positive_name}' to '{anchor_name}'")
                    renamed_count += 1
                except OSError as e:
                    print(f"  Error renaming positive file '{old_positive_name}' to '{anchor_name}': {e}")

        # Rename negative file
        if old_negative_name != anchor_name: # Only rename if name is different
            if os.path.exists(new_negative_path):
                print(f"  Skipping negative rename for '{anchor_name}': Target path '{new_negative_path}' already exists. Move/delete it first if you want to replace.")
            else:
                try:
                    os.rename(old_negative_path, new_negative_path)
                    print(f"  Renamed negative: '{old_negative_name}' to '{anchor_name}'")
                    renamed_count += 1
                except OSError as e:
                    print(f"  Error renaming negative file '{old_negative_name}' to '{anchor_name}': {e}")

    print(f"\nFinished renaming process for '{base_dir}'. Total files renamed: {renamed_count}")

# Example Usage:
# You would typically run this for your training and validation datasets.
# Uncomment and modify paths as needed.
# Make sure to run the cell with the `rename_triplet_files` function definition first.
print("Renaming files for training data...")
rename_triplet_files("./train/Footwear/Men/")

print("\nRenaming files for validation data...")
rename_triplet_files("./test/Footwear/Men/")


Renaming files for training data...

Attempting to rename 0 triplets in './train/Footwear/Men/'...

Finished renaming process for './train/Footwear/Men/'. Total files renamed: 0

Renaming files for validation data...

Attempting to rename 0 triplets in './test/Footwear/Men/'...

Finished renaming process for './test/Footwear/Men/'. Total files renamed: 0


In [21]:
import os
from PIL import Image
import random
from torch.utils.data import Dataset
from torchvision import transforms

class TripletDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        self.anchor_dir = os.path.join(root_dir, 'anchor')
        self.positive_dir = os.path.join(root_dir, 'positive')
        self.negative_dir = os.path.join(root_dir, 'negative')

        if not os.path.isdir(self.anchor_dir) or \
           not os.path.isdir(self.positive_dir) or \
           not os.path.isdir(self.negative_dir):
            raise RuntimeError(f"Expected 'anchor', 'positive', and 'negative' subdirectories in {root_dir}")

        self.triplets = self._find_triplets()

        if not self.triplets:
            raise RuntimeError(f"Found 0 valid triplets in {root_dir}. Ensure corresponding images exist in anchor, positive, and negative subdirectories.")

    def _find_triplets(self):
        triplets = []
        anchor_images = sorted([f for f in os.listdir(self.anchor_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))])

        for img_name in anchor_images:
            positive_path = os.path.join(self.positive_dir, img_name)
            negative_path = os.path.join(self.negative_dir, img_name)

            if os.path.exists(positive_path) and os.path.exists(negative_path):
                triplets.append((os.path.join(self.anchor_dir, img_name), positive_path, negative_path))
        return triplets

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        anchor_path, positive_path, negative_path = self.triplets[idx]

        anchor_img = Image.open(anchor_path).convert("RGB")
        positive_img = Image.open(positive_path).convert("RGB")
        negative_img = Image.open(negative_path).convert("RGB")

        if self.transform:
            anchor_img = self.transform(anchor_img)
            positive_img = self.transform(positive_img)
            negative_img = self.transform(negative_img)

        return anchor_img, positive_img, negative_img

In [22]:
# Update the dataset and dataloader creation with the new TripletDataset

try:
    train_dataset = TripletDataset("./train/Footwear/Men/", transform=transform)
    val_dataset = TripletDataset("./test/Footwear/Men/", transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)
    print("Triplet Datasets and DataLoaders created successfully!")
except RuntimeError as e:
    print(f"Error creating Triplet Dataset: {e}")
    print("Please ensure your root directories (e.g., ./train/Footwear/Men/) contain 'anchor', 'positive', and 'negative' subdirectories, and that corresponding image files exist in each.")
    print("Example: ./train/Footwear/Men/anchor/image1.jpg, ./train/Footwear/Men/positive/image1.jpg, ./train/Footwear/Men/negative/image1.jpg")

Error creating Triplet Dataset: Found 0 valid triplets in ./train/Footwear/Men/. Ensure corresponding images exist in anchor, positive, and negative subdirectories.
Please ensure your root directories (e.g., ./train/Footwear/Men/) contain 'anchor', 'positive', and 'negative' subdirectories, and that corresponding image files exist in each.
Example: ./train/Footwear/Men/anchor/image1.jpg, ./train/Footwear/Men/positive/image1.jpg, ./train/Footwear/Men/negative/image1.jpg


In [23]:
import os
import shutil

# Path to the .ipynb_checkpoints directory that might be causing issues
checkpoint_dir_train = "./train/Footwear/Men/.ipynb_checkpoints"
checkpoint_dir_val = "./test/Footwear/Men/.ipynb_checkpoints"

# Check if the directory exists and remove it
if os.path.exists(checkpoint_dir_train) and os.path.isdir(checkpoint_dir_train):
    shutil.rmtree(checkpoint_dir_train)
    print(f"Removed problematic directory: {checkpoint_dir_train}")

if os.path.exists(checkpoint_dir_val) and os.path.isdir(checkpoint_dir_val):
    shutil.rmtree(checkpoint_dir_val)
    print(f"Removed problematic directory: {checkpoint_dir_val}")

try:
    train_dataset = TripletDataset("./train/Footwear/Men/", transform=transform)
    val_dataset = TripletDataset("./test/Footwear/Men/", transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)
    print("Triplet Datasets and DataLoaders created successfully!")
except RuntimeError as e:
    print(f"Error creating Triplet Dataset: {e}")
    print("Please ensure your root directories (e.g., ./train/Footwear/Men/) contain 'anchor', 'positive', and 'negative' subdirectories, and that corresponding image files exist in each.")
    print("Example: ./train/Footwear/Men/anchor/image1.jpg, ./train/Footwear/Men/positive/image1.jpg, ./train/Footwear/Men/negative/image1.jpg")

# ===== 3. Model, Loss, Optimizer =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = get_resnet50_encoder(embedding_dim=256, pretrained=True, train_backbone=True).to(device)

# Example: contrastive learning loss (InfoNCE, Triplet, etc.)
# Here we use TripletMarginLoss for demonstration
criterion = nn.TripletMarginLoss(margin=1.0, p=2)

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

Error creating Triplet Dataset: Found 0 valid triplets in ./train/Footwear/Men/. Ensure corresponding images exist in anchor, positive, and negative subdirectories.
Please ensure your root directories (e.g., ./train/Footwear/Men/) contain 'anchor', 'positive', and 'negative' subdirectories, and that corresponding image files exist in each.
Example: ./train/Footwear/Men/anchor/image1.jpg, ./train/Footwear/Men/positive/image1.jpg, ./train/Footwear/Men/negative/image1.jpg


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [24]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, desc="Training"):
        # For Triplet loss, you need (anchor, positive, negative) samples
        # Here we assume you have a custom dataset that returns them
        # This is just a placeholder
        anchor, positive, negative = batch  # Replace with your triplet dataset
        anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)

        optimizer.zero_grad()
        emb_a = model(anchor)
        emb_p = model(positive)
        emb_n = model(negative)

        loss = criterion(emb_a, emb_p, emb_n)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [25]:
def validate_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in tqdm(loader, desc="Validating"):
            anchor, positive, negative = batch
            anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)

            emb_a = model(anchor)
            emb_p = model(positive)
            emb_n = model(negative)

            loss = criterion(emb_a, emb_p, emb_n)
            total_loss += loss.item()

    return total_loss / len(loader)

In [26]:
EPOCHS = 10
for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    val_loss = validate_epoch(model, val_loader, criterion)
    print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f}")

# ===== 7. Save the trained encoder =====
torch.save(model.state_dict(), "resnet50_encoder.pth")
print("Model saved as resnet50_encoder.pth")

NameError: name 'train_loader' is not defined

In [ ]:
import matplotlib.pyplot as plt

# Plotting the training and validation loss
plt.figure(figsize=(10, 6))
plt.plot(range(1, EPOCHS + 1), train_losses, label='Training Loss')
plt.plot(range(1, EPOCHS + 1), val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss Over Epochs')
plt.legend()
plt.grid(True)
plt.show()
